<a href="https://colab.research.google.com/github/NeuromatchAcademy/course-content/blob/main/tutorials/W1D2_ModelFitting/student/W1D2_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NeuromatchAcademy/course-content/main/tutorials/W1D2_ModelFitting/student/W1D2_Intro.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# はじめに

## 概要

本日はモデルを実験データと照らし合わせる方法についてお話しします。私のモデルは被験者の行動や神経活動を捉えているでしょうか？データは私のモデルを他の代替モデルと比べて支持しているでしょうか？モデルのどの要素が必要でしょうか？モデルのパラメータは2つの被験者集団間で系統的に異なっているでしょうか？これらの疑問に答えるための基本的な概念とツールを、イントロダクションでの全体概要から始めて解説します。チュートリアル1と2ではデータから単純な回帰モデルのパラメータを推定する方法を学び、チュートリアル3ではこれらの値の不確実性を推定する方法を学びます。続いて、異なる複雑さのモデルからデータを最もよく説明するモデルを選択する方法をチュートリアル4〜6で学びます。最後に、アウトロでこれらの手法を実際の研究例で示します。

モデルのフィッティングと比較は神経科学におけるデータ解析の基本中の基本です。モデルタイプの日には、神経科学で関心のあるさまざまな種類のモデルについて学びました。ここでは、あらゆるタイプのモデルのフィッティングと比較に共通して使える一般的な概念と手法を学びます。これは非常に役立つ内容です！これらのツールはGLM、潜在モデル、深層ネットワーク、動的モデル、意思決定モデル、強化学習モデルなどを扱う際にも再び使います。さらに、本日は連続変数（例：BOLD活動）を従属変数とする典型的な線形回帰モデルを取り上げ、学んだ概念や手法の説明に用います。GLMの日には、従属変数が二値（例：選択）や整数（例：スパイク数）である場合の回帰モデルの一般化を学びます。

ほぼすべての統計的・データ解析手法は、明示的または暗黙的に何らかのモデルをデータにフィットさせることに依存しています。本日学ぶ概念とツールは、行動や神経活動がどのように形成されるかについての仮説を検証するために不可欠です。典型的な方法は、仮説を体現した計算的（確率的）モデルと1つ以上の対照モデルを定式化し、それぞれのモデルを実験データ（例えば、ある被験者の選択パターンや記録された1つのニューロンのスパイク活動）にフィットさせます。フィットしたモデルをシミュレーションすることで、モデルがデータ中の関心のある効果を確かに捉えているかを検証できます。次に、モデル比較の手法を用いて、主要モデルまたは対照モデルのどちらがデータをよりよく説明しているかを判断します。また、モデルのパラメータが実験条件や被験者集団間で変化しているかを評価することも可能です。

## 前提知識

本日の内容とチュートリアルでは、ベクトルと行列（[W0D3 T1](https://precourse.neuromatch.io/tutorials/W0D3_LinearAlgebra/student/W0D3_Tutorial1.html) & [T2](https://precourse.neuromatch.io/tutorials/W0D3_LinearAlgebra/student/W0D3_Tutorial2.html)）、確率分布（[W0D5 T1](https://precourse.neuromatch.io/tutorials/W0D5_Statistics/student/W0D5_Tutorial1.html)）、および尤度（[W0D5 T2](https://precourse.neuromatch.io/tutorials/W0D5_Statistics/student/W0D5_Tutorial2.html)）を使用します。必要に応じてこれらの資料を復習してください！

In [ ]:
# @title Install and import feedback gadget

!pip3 install vibecheck datatops --quiet

from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",  # No text prompt
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "neuromatch_cn",
            "user_key": "y1x3mpx5",
        },
    ).render()


feedback_prefix = "W1D2_Intro"

## ビデオ

In [ ]:
# @markdown
from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
  tab_contents = []
  for i, video_id in enumerate(video_ids):
    out = widgets.Output()
    with out:
      if video_ids[i][0] == 'Youtube':
        video = YouTubeVideo(id=video_ids[i][1], width=W,
                             height=H, fs=fs, rel=0)
        print(f'Video available at https://youtube.com/watch?v={video.id}')
      else:
        video = PlayVideo(id=video_ids[i][1], source=video_ids[i][0], width=W,
                          height=H, fs=fs, autoplay=False)
        if video_ids[i][0] == 'Bilibili':
          print(f'Video available at https://www.bilibili.com/video/{video.id}')
        elif video_ids[i][0] == 'Osf':
          print(f'Video available at https://osf.io/{video.id}')
      display(video)
    tab_contents.append(out)
  return tab_contents


video_ids = [('Youtube', '9JfXKmVB6qc'), ('Bilibili', 'BV1BX4y1w7oc')]
tab_contents = display_videos(video_ids, W=854, H=480)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
  tabs.set_title(i, video_ids[i][0])
display(tabs)

## スライド

In [ ]:
# @markdown
from IPython.display import IFrame
link_id = "sqcvz"
print(f"If you want to download the slides: https://osf.io/download/{link_id}/")
IFrame(src=f"https://mfr.ca-1.osf.io/render?url=https://osf.io/{link_id}/?direct%26mode=render%26action=download%26mode=render", width=854, height=480)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Intro_Video")